# Natural Language Processing

## Introduction

In [ ]:
import numpy as np
import pandas as pd

from itables import show
import pprint
import os
import string
from tqdm import tqdm

from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn import manifold
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import LatentDirichletAllocation

from transformers import pipeline
from staticvectors import StaticVectors

import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

from ind5003 import nlp

## Definitions
## Overview of Applications
## Text Pre-processing
### Example: Wine reviews dataset

In [ ]:
rng = np.random.default_rng(5001)

wine_reviews = pd.read_csv("data/winemag-data-130k-v2.csv", index_col=0)
wine_reviews.drop_duplicates(inplace=True)

In [ ]:
pp = pprint.PrettyPrinter(indent=4, compact=True,)
for x in rng.choice(wine_reviews.description, size=5):
    pp.pprint(x)

### Pre-processing Text with Python

In [ ]:
def custom_preprocessor(x, min_length = 3, stop_words = []):
    output = x.lower()
    output = output.translate(str.maketrans('', '', string.punctuation + '0123456789'))
    tokens = word_tokenize(output)
    output_list = [y for y in tokens if y not in stop_words and len(y) > min_length]
    return output_list

In [ ]:
en_stop_words = stopwords.words('english')
all_review_strings = wine_reviews.description.values
all_strings_tokenized = [custom_preprocessor(x, 3, en_stop_words) for x in all_review_strings]

In [ ]:
pp.pprint(wine_reviews.description.values[233])

In [ ]:
pp.pprint(all_strings_tokenized[233])

### Example: Stemming versus lemmatizing

In [ ]:
demo_sentence = 'Cats and ponies have a meeting'.split()
demo_sentence

In [ ]:
porter = PorterStemmer()
[porter.stem(x) for x in demo_sentence]

In [ ]:
#| eval: false
import nltk
nltk.download('wordnet')

In [ ]:
wn = WordNetLemmatizer()
[wn.lemmatize(x) for x in demo_sentence]

In [ ]:
preprocessed_corpus = [[wn.lemmatize(w) for w in dd ] for dd in all_strings_tokenized]

## Representation of Text {#sec-04-numeric-rep-text}
### Sparse embeddings with Tf-idf
### Example: Term frequencies

In [ ]:
raw_docs =[
  "Here are some very simple basic sentences.", 
  "They won’t be very interesting , I’m afraid. ",  
  """
  The point of these basic examples is to learn how basic text  
  counting works on *very simple* data, so that we are not afraid when  
  it comes to larger text documents. The sentences are here just to provide words.
  """] 

In [ ]:
vectorizer1 = CountVectorizer(stop_words='english', min_df=1) 
raw_docs_dtm = vectorizer1.fit_transform(raw_docs)

print(pd.DataFrame(raw_docs_dtm.toarray(),  
   columns=vectorizer1.get_feature_names_out()).iloc[:, :10])

### Example: Tf-idf computation

In [ ]:
vectorizer2 = TfidfVectorizer(stop_words='english', norm=None)
raw_docs_tfidf_no_norm = vectorizer2.fit_transform(raw_docs)

print(pd.DataFrame(raw_docs_tfidf_no_norm.toarray(), 
  columns=list(vectorizer2.get_feature_names_out())).iloc[:, :10].round(3))

In [ ]:
vectorizer3 = TfidfVectorizer(stop_words='english')
raw_docs_tfidf_norm = vectorizer3.fit_transform(raw_docs)

print(pd.DataFrame(raw_docs_tfidf_norm.toarray(), 
  columns=list(vectorizer3.get_feature_names_out())).iloc[:, :10].round(3))

### Cosine similarity
### Dense Embeddings {#sec-04-dense}
### Example: Glove dense embeddings

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
word_vectors = StaticVectors("neuml/glove-6B")

In [ ]:
nbrs = NearestNeighbors(n_neighbors=5, 
                        metric='cosine', algorithm='auto').fit(word_vectors.vectors)

In [ ]:
nlp.word_analogy(word_vectors, nbrs, 'man', 'king', 'woman')
#word_analogy(word_vectors, nbrs, 'switzerland', 'swiss', 'cambodia')

## Visualisation with t-SNE {#sec-04-t-SNE}

In [ ]:
nn = 1000
glove_1e3 = word_vectors.vectors[:nn, :]
labels = pd.Series(list(word_vectors.tokens.keys())[:nn])

tsne1 = manifold.TSNE(n_components=2, init="random", perplexity=10, 
                      metric='cosine', verbose=0, max_iter=5000, 
                      random_state=222)
glove_transformed = tsne1.fit_transform(glove_1e3)

df2 = pd.DataFrame(glove_transformed, columns=['x','y'])
df2['labels'] = labels

In [ ]:
#| fig-align: center
fig = px.scatter(df2, x='x', y='y', text='labels', width=1024, height=960)
fig.update_traces(textposition='top center')
fig.show()

In [ ]:
#| fig-align: center
#| fig-cap: t-SNE plot of GloVe embedding
#| label: fig-tsne-glove
#| fig-pos: 'ht'

plt.figure(figsize=(8, 4))
sns.scatterplot(data=df2, x="x", y="y")

## Neural Language Models
## Applications {#sec-nlp-applications}
### Sentiment Analysis

In [ ]:
classifier = pipeline("sentiment-analysis", 
  model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
  
classifier(["I love this course!", "I absolutely detest this course."])

### Example: Wine reviews sentiments

In [ ]:
tmp_df = wine_reviews.head(0).copy()

for x,vv in wine_reviews.groupby(wine_reviews.variety):
    grp_len = vv.shape[0]
    if(grp_len >= 20):
        vv = vv.sample(n=20, random_state=99)
    tmp_df = pd.concat([tmp_df, vv], ignore_index=True)
    
review_list = list(tmp_df.description)

In [ ]:
tmp_df['score'] = 0.00
tmp_df['label'] = ''

for i, rr in tqdm(enumerate(review_list), total=len(review_list), desc="Classifying reviews"):
    tmp = classifier(rr)[0]
    tmp_df.loc[i, 'score'] = tmp['score']
    tmp_df.loc[i, 'label'] = tmp['label']

In [ ]:
sent_counts = pd.crosstab(tmp_df.variety, tmp_df.label, margins=True)
sent_counts['proportion'] = sent_counts.POSITIVE/sent_counts.All

In [ ]:
show(sent_counts)

In [ ]:
sent_counts.head()

In [ ]:
for x in wine_reviews[wine_reviews.variety == 'Tempranillo Blanco'].description.values:
    pp.pprint(x) 

In [ ]:
tmp_df[tmp_df.variety == 'Tempranillo Blanco'][['description', 'label']]

### Information Retrieval
### Example: Wine reviews information retrieval

In [ ]:
# IR retrieval use-case:
wine_tfidf = TfidfVectorizer(stop_words='english')
X2 = wine_tfidf.fit_transform(wine_reviews.description)

wine_nbrs = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='auto').fit(X2)

In [ ]:
awc = wine_tfidf.transform(['acidic white chardonnay'])
distances, indices = wine_nbrs.kneighbors(awc, n_neighbors=8)
for x in indices[0]:
    pp.pprint(all_review_strings[x]) 

### Topic Modeling
### Example: Wine reviews topic modeling

In [ ]:
en_stop_words = en_stop_words + ["flavors", "years", "wine", "drink", "fruits", 
                                 "fruit", "finish", "aromas", "palate", "feels", 
                                 "notes", "oak", "offers", "like", "bottling", 
                                 "nose", "shows"]

# Step 3: Train LDA model
lda_obj = LatentDirichletAllocation(
    n_components=5,           # Number of topics
    random_state=41,
    max_iter=20,
    learning_method='batch',
    evaluate_every=-1, verbose=0, n_jobs=2
)

wine_tfidf = TfidfVectorizer(stop_words=en_stop_words)
X2 = wine_tfidf.fit_transform(wine_reviews.description)
lda_obj.fit(X2);

In [ ]:
feature_names = wine_tfidf.get_feature_names_out()

n_top = 10
rows = []
for i, topic in enumerate(lda_obj.components_):
    top_words = [feature_names[j.item()] for j in topic.argsort()[-n_top:][::-1]]
    rows.append(top_words)

[','.join(x) for x in rows]

In [ ]:
#| eval: false
%pip install --no-deps pyLDAvis

In [ ]:
#| eval: false
import pyLDAvis
import pyLDAvis.lda_model as lda_model

vis = lda_model.prepare(lda_obj, X2, wine_tfidf)
pyLDAvis.display(vis)

## Interpretation of Neural Models
## References
### Video explainers
### Website References
## Exercises